In [1]:
import pandas as pd
import cupy as cp
import cudf
import cuml
import torch
import gc
import time
from sklearn.datasets import make_blobs
from cuml.cluster import HDBSCAN

In [2]:
df_pandas = pd.read_csv('household_power_consumption_not_null.csv', parse_dates=[['Date', 'Time']],
                       date_format = {'Date': '%d/%m/%Y', 
                                      'Time': '%H:%M:%S'},
                       dayfirst = True)
df_pandas

/tmp/ipykernel_9597/4030125953.py:1: FutureWarning: Support for nested sequences for 'parse_dates' in pd.read_csv is deprecated. Combine the desired columns with pd.to_datetime after parsing instead.
  df_pandas = pd.read_csv('household_power_consumption_not_null.csv', parse_dates=[['Date', 'Time']],


,Date_Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0
...,...,...,...,...,...,...,...,...
2049275,2010-11-26 20:58:00,0.946,0.000,240.43,4.0,0.0,0.0,0.0
2049276,2010-11-26 20:59:00,0.944,0.000,240.00,4.0,0.0,0.0,0.0
2049277,2010-11-26 21:00:00,0.938,0.000,239.82,3.8,0.0,0.0,0.0
2049278,2010-11-26 21:01:00,0.934,0.000,239.70,3.8,0.0,0.0,0.0


In [3]:
df_cudf_1 = cudf.DataFrame(df_pandas)
df_cudf_1

,Date_Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0
...,...,...,...,...,...,...,...,...
2049275,2010-11-26 20:58:00,0.946,0.000,240.43,4.0,0.0,0.0,0.0
2049276,2010-11-26 20:59:00,0.944,0.000,240.00,4.0,0.0,0.0,0.0
2049277,2010-11-26 21:00:00,0.938,0.000,239.82,3.8,0.0,0.0,0.0
2049278,2010-11-26 21:01:00,0.934,0.000,239.70,3.8,0.0,0.0,0.0


In [4]:
df_cudf_1 = df_cudf_1.astype(float)
df_cudf_1

,Date_Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,1.166290e+18,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,1.166290e+18,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,1.166290e+18,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,1.166290e+18,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,1.166290e+18,3.666,0.528,235.68,15.8,0.0,1.0,17.0
...,...,...,...,...,...,...,...,...
2049275,1.290805e+18,0.946,0.000,240.43,4.0,0.0,0.0,0.0
2049276,1.290805e+18,0.944,0.000,240.00,4.0,0.0,0.0,0.0
2049277,1.290805e+18,0.938,0.000,239.82,3.8,0.0,0.0,0.0
2049278,1.290805e+18,0.934,0.000,239.70,3.8,0.0,0.0,0.0


In [5]:
class Clustering(object):

    def __init__(self, dataset):
        self.dataset = dataset.copy().reset_index(drop = True)

    def HDBSCAN(self):
        global HDBSCAN_global
        global labels_global
        global cluster_centers_global

        HDBSCAN_float = cuml.cluster.hdbscan.HDBSCAN(min_cluster_size=5, min_samples=None, 
                                                                cluster_selection_epsilon=0.0, max_cluster_size=0, 
                                                                metric='euclidean', alpha=1.0, p=None, 
                                                                cluster_selection_method='eom', allow_single_cluster=False, 
                                                                gen_min_span_tree=False, verbose=False, output_type=None, 
                                                                prediction_data=False, build_algo='brute_force', 
                                                                build_kwds=None, device_ids=None)
        HDBSCAN = HDBSCAN_float.fit(self.dataset)
        HDBSCAN_global = HDBSCAN

        labels = HDBSCAN_float.labels_
        labels_global = labels

        
    def main(self):
        st = time.time()
        self.HDBSCAN()
        et = time.time()
        elapsed_time = et - st
        print('Execution time:', elapsed_time, 'seconds')

In [6]:
clust = Clustering(df_cudf_1)

In [7]:
clust.main()

Execution time: 1008.1384177207947 seconds


In [9]:
HDBSCAN_global

HDBSCAN()

In [10]:
labels_global

0          0
1          0
2          0
3          0
4          0
          ..
2049275    1
2049276    1
2049277    1
2049278    1
2049279    1
Length: 2049280, dtype: int64